[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/ia_p26/blob/main/clase/19_cadenas_de_markov/notebooks/aplicaciones/04_mercados_financieros.ipynb)


# Mercados Financieros: Cadenas de Markov para Regímenes

**Módulo 19 — Cadenas de Markov · Notebook 04 (Aplicación)**


In [ ]:
import sys
if "google.colab" in sys.modules:
    !pip install -q numpy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

COLORS = {
    "blue": "#2E86AB", "red": "#E94F37", "green": "#27AE60",
    "gray": "#7F8C8D", "orange": "#F39C12", "purple": "#8E44AD",
    "dark": "#2C3E50", "teal": "#1ABC9C", "light": "#ECF0F1",
}

np.random.seed(42)


## Parte 1: Datos reales de mercado

Modelamos los mercados financieros como **cadenas de Markov sobre regímenes**.
Un mercado fluctúa entre tres estados:

| Régimen | Descripción |
|:-------:|:------------|
| **Bull** (alcista)  | Rendimientos positivos sostenidos |
| **Bear** (bajista)  | Caídas pronunciadas y alta volatilidad |
| **Flat** (lateral)  | Rendimientos cercanos a cero |

La idea central es que la **dinámica de transición** entre estos regímenes
se puede capturar con una matriz estocástica $P$, y el **teorema ergódico**
nos dice cuánto tiempo pasará el mercado en cada estado a largo plazo.


In [ ]:
# ── S&P 500: datos basados en parámetros históricos (1965-2024) ──────────
np.random.seed(42)
n_sp = 720                          # ~60 años × 12 meses
regime = 0                          # comienza en Bull

P_gen = np.array([
    [0.85, 0.08, 0.07],            # Bull  → Bull / Bear / Flat
    [0.10, 0.75, 0.15],            # Bear  → Bull / Bear / Flat
    [0.15, 0.10, 0.75],            # Flat  → Bull / Bear / Flat
])

means = [0.015, -0.02, 0.003]      # rendimiento mensual medio
stds  = [0.03,   0.055, 0.025]     # volatilidad mensual

returns_sp, true_regimes_sp = [], []
for _ in range(n_sp):
    true_regimes_sp.append(regime)
    returns_sp.append(np.random.normal(means[regime], stds[regime]))
    regime = np.random.choice(3, p=P_gen[regime])

returns_sp = np.array(returns_sp)
true_regimes_sp = np.array(true_regimes_sp)
years_sp = np.linspace(1965, 2025, n_sp)

# ── IPC México: parámetros con mayor volatilidad (1994-2024) ─────────────
np.random.seed(123)
n_ipc = 360                         # ~30 años × 12 meses
regime = 0

P_gen_mx = np.array([
    [0.80, 0.10, 0.10],
    [0.12, 0.70, 0.18],
    [0.15, 0.12, 0.73],
])

means_mx = [0.018, -0.025, 0.005]
stds_mx  = [0.045,  0.07,  0.035]

returns_ipc, true_regimes_ipc = [], []
for _ in range(n_ipc):
    true_regimes_ipc.append(regime)
    returns_ipc.append(np.random.normal(means_mx[regime], stds_mx[regime]))
    regime = np.random.choice(3, p=P_gen_mx[regime])

returns_ipc = np.array(returns_ipc)
true_regimes_ipc = np.array(true_regimes_ipc)
years_ipc = np.linspace(1994, 2024, n_ipc)

print(f"S&P 500 : {n_sp} meses generados  (media = {returns_sp.mean():.4f})")
print(f"IPC MX  : {n_ipc} meses generados (media = {returns_ipc.mean():.4f})")


In [ ]:
# ── Índice de precios acumulado (base 100) ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

price_sp  = 100 * np.cumprod(1 + returns_sp)
price_ipc = 100 * np.cumprod(1 + returns_ipc)

axes[0].plot(years_sp, price_sp, color=COLORS["blue"], lw=1.2)
axes[0].set_title("S&P 500 — Índice sintético (base 100)", fontsize=13)
axes[0].set_xlabel("Año")
axes[0].set_ylabel("Índice")
axes[0].set_yscale("log")

axes[1].plot(years_ipc, price_ipc, color=COLORS["teal"], lw=1.2)
axes[1].set_title("IPC México — Índice sintético (base 100)", fontsize=13)
axes[1].set_xlabel("Año")
axes[1].set_ylabel("Índice")
axes[1].set_yscale("log")

for ax in axes:
    ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()


## Parte 2: Clasificación y estimación de la matriz de transición

Clasificamos cada mes según su rendimiento:

| Régimen | Criterio |
|:-------:|:---------|
| **Bull**  | rendimiento $> 2\%$ |
| **Bear**  | rendimiento $< -2\%$ |
| **Flat**  | $-2\% \leq$ rendimiento $\leq 2\%$ |

A partir de esta clasificación contamos las transiciones observadas y
estimamos la matriz $\hat{P}$ por **máxima verosimilitud** (frecuencias relativas).


In [ ]:
def classify_regimes(returns, bull_thresh=0.02, bear_thresh=-0.02):
    """Clasifica rendimientos en regímenes: 0=Bull, 1=Bear, 2=Flat."""
    regimes = np.zeros(len(returns), dtype=int)
    regimes[returns > bull_thresh]  = 0   # Bull
    regimes[returns < bear_thresh]  = 1   # Bear
    regimes[(returns >= bear_thresh) & (returns <= bull_thresh)] = 2  # Flat
    return regimes


def estimate_transition_matrix(regimes, n_states=3):
    """Estima la matriz de transición por frecuencias relativas."""
    counts = np.zeros((n_states, n_states))
    for t in range(len(regimes) - 1):
        counts[regimes[t], regimes[t + 1]] += 1
    # Normalizar filas
    row_sums = counts.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1          # evitar división por cero
    P_hat = counts / row_sums
    return P_hat, counts


# ── S&P 500 ──────────────────────────────────────────────────────────────
regimes_sp = classify_regimes(returns_sp)
P_sp, counts_sp = estimate_transition_matrix(regimes_sp)

labels = ["Bull", "Bear", "Flat"]
print("Matriz de transición estimada — S&P 500")
print("-" * 42)
header = "         " + "".join(f"{l:>10}" for l in labels)
print(header)
for i, row_label in enumerate(labels):
    row = f"{row_label:>8} " + "".join(f"{P_sp[i, j]:10.4f}" for j in range(3))
    print(row)

print(f"\nSuma de filas: {P_sp.sum(axis=1)}")
print(f"\nConteo de transiciones:\n{counts_sp.astype(int)}")


### Regímenes verdaderos vs. estimados

Es importante notar que los regímenes que detectamos con umbrales fijos (retorno > 2% = Alcista, < -2% = Bajista) son una **aproximación cruda** de los regímenes "verdaderos" del mercado. En nuestros datos sintéticos, conocemos los regímenes reales porque los generamos con `P_gen` — pero en la realidad nunca observamos el régimen directamente, solo los retornos.

Esta brecha entre regímenes ocultos y observaciones ruidosas es exactamente lo que motiva los **Modelos Ocultos de Markov (HMMs)**: modelos donde los estados (regímenes) son latentes y solo observamos señales ruidosas (retornos) que dependen del estado oculto. Los HMMs son el siguiente paso natural después de las cadenas de Markov observables.

In [ ]:
# ── Rendimientos mensuales con regímenes coloreados ──────────────────────
fig, ax = plt.subplots(figsize=(14, 5))

regime_colors = {0: COLORS["green"], 1: COLORS["red"], 2: COLORS["gray"]}
regime_labels = {0: "Bull", 1: "Bear", 2: "Flat"}

for i in range(len(returns_sp)):
    ax.bar(years_sp[i], returns_sp[i],
           width=(years_sp[1] - years_sp[0]) * 0.9,
           color=regime_colors[regimes_sp[i]], alpha=0.7)

ax.axhline(0, color="black", lw=0.8)
ax.set_title("S&P 500 — Rendimientos mensuales coloreados por régimen", fontsize=13)
ax.set_xlabel("Año")
ax.set_ylabel("Rendimiento mensual")

patches = [mpatches.Patch(color=regime_colors[k], label=regime_labels[k])
           for k in [0, 1, 2]]
ax.legend(handles=patches, loc="upper left")
ax.set_xlim(years_sp[0], years_sp[-1])
fig.tight_layout()
plt.show()


In [ ]:
def stationary_distribution(P):
    """Calcula la distribución estacionaria π resolviendo πP = π.

    Usamos el método del vector propio izquierdo con valor propio 1.
    """
    n = P.shape[0]
    # πP = π  ⟺  P^T π^T = π^T
    eigenvalues, eigenvectors = np.linalg.eig(P.T)
    # Buscar valor propio ≈ 1
    idx = np.argmin(np.abs(eigenvalues - 1.0))
    pi = np.real(eigenvectors[:, idx])
    pi = pi / pi.sum()               # normalizar
    return pi


pi_sp = stationary_distribution(P_sp)

print("Distribución estacionaria — S&P 500")
print("-" * 40)
for i, label in enumerate(["Bull", "Bear", "Flat"]):
    obs_frac = (regimes_sp == i).mean()
    print(f"  π({label:>4}) = {pi_sp[i]:.4f}   (observado: {obs_frac:.4f})")


In [ ]:
# ── IPC México ───────────────────────────────────────────────────────────
regimes_ipc = classify_regimes(returns_ipc)
P_ipc, counts_ipc = estimate_transition_matrix(regimes_ipc)
pi_ipc = stationary_distribution(P_ipc)

print("Matriz de transición estimada — IPC México")
print("-" * 42)
header = "         " + "".join(f"{l:>10}" for l in labels)
print(header)
for i, row_label in enumerate(labels):
    row = f"{row_label:>8} " + "".join(f"{P_ipc[i, j]:10.4f}" for j in range(3))
    print(row)

print()

# ── Comparación ──────────────────────────────────────────────────────────
print("=" * 60)
print("COMPARACIÓN DE MATRICES DE TRANSICIÓN")
print("=" * 60)
print(f"\n{'Transición':<16} {'S&P 500':>10} {'IPC MX':>10} {'Diferencia':>12}")
print("-" * 50)
for i in range(3):
    for j in range(3):
        t_label = f"{labels[i]}→{labels[j]}"
        diff = P_ipc[i, j] - P_sp[i, j]
        print(f"{t_label:<16} {P_sp[i,j]:10.4f} {P_ipc[i,j]:10.4f} {diff:+12.4f}")

print(f"\n{'Distribución estacionaria':}")
print(f"{'Régimen':<10} {'S&P 500':>10} {'IPC MX':>10}")
print("-" * 32)
for i, label in enumerate(labels):
    print(f"{label:<10} {pi_sp[i]:10.4f} {pi_ipc[i]:10.4f}")


## Parte 3: Simulación y comparación

Simulamos **1000 trayectorias** de regímenes usando la matriz estimada del
S&P 500 y comparamos las estadísticas simuladas con los datos observados.
Esto valida que el modelo de Markov captura razonablemente la dinámica real.


In [ ]:
def simulate_regime_trajectories(P, n_steps, n_sims, pi_start=None):
    """Simula n_sims trayectorias de regímenes de longitud n_steps.

    Parametros
    ----------
    P : ndarray (k, k)
        Matriz de transición.
    n_steps : int
        Longitud de cada trayectoria.
    n_sims : int
        Número de simulaciones.
    pi_start : ndarray, opcional
        Distribución inicial. Si None, usa la estacionaria.

    Retorna
    -------
    trajectories : ndarray (n_sims, n_steps)
    """
    k = P.shape[0]
    if pi_start is None:
        pi_start = stationary_distribution(P)

    trajectories = np.zeros((n_sims, n_steps), dtype=int)
    trajectories[:, 0] = np.array(
        [np.random.choice(k, p=pi_start) for _ in range(n_sims)]
    )

    for t in range(1, n_steps):
        for s in range(n_sims):
            trajectories[s, t] = np.random.choice(k, p=P[trajectories[s, t - 1]])

    return trajectories


np.random.seed(42)
n_sims = 1000
trajs = simulate_regime_trajectories(P_sp, n_steps=n_sp, n_sims=n_sims)
print(f"Simuladas {n_sims} trayectorias de {n_sp} pasos")
print(f"Forma de la matriz de trayectorias: {trajs.shape}")


### Interpretación estadística

Vamos a comparar estadísticas de 1000 trayectorias simuladas con las de **una sola** trayectoria observada. Es importante recordar que la trayectoria observada es una realización particular con su propia varianza muestral — no es "la verdad", sino una muestra. Las diferencias entre valores observados y esperados (teóricos) son naturales y esperables. Lo que buscamos es que los valores observados caigan dentro del rango de variación de las simulaciones.

In [ ]:
def regime_durations(seq, regime):
    """Calcula las duraciones de rachas consecutivas de un régimen."""
    durations = []
    count = 0
    for s in seq:
        if s == regime:
            count += 1
        else:
            if count > 0:
                durations.append(count)
            count = 0
    if count > 0:
        durations.append(count)
    return durations


# ── Estadísticas observadas ──────────────────────────────────────────────
real_stats = {}
for r, label in enumerate(labels):
    durs = regime_durations(regimes_sp, r)
    real_stats[label] = {
        "mean_dur": np.mean(durs) if durs else 0,
        "frac": (regimes_sp == r).mean(),
        "max_bear": max(regime_durations(regimes_sp, 1)) if r == 1 else None,
    }

# ── Estadísticas simuladas ──────────────────────────────────────────────
sim_mean_durs = {r: [] for r in range(3)}
sim_fracs     = {r: [] for r in range(3)}
sim_max_bear  = []

for s in range(n_sims):
    traj = trajs[s]
    for r in range(3):
        durs = regime_durations(traj, r)
        sim_mean_durs[r].append(np.mean(durs) if durs else 0)
        sim_fracs[r].append((traj == r).mean())
    bear_durs = regime_durations(traj, 1)
    sim_max_bear.append(max(bear_durs) if bear_durs else 0)

# ── Tabla comparativa ────────────────────────────────────────────────────
print("COMPARACIÓN: DATOS OBSERVADOS vs SIMULACIÓN (1000 trayectorias)")
print("=" * 75)

print(f"\n{'Métrica':<30} {'Observado':>12} {'Sim media':>12} {'Sim std':>10} {'π':>8}")
print("-" * 75)

for r, label in enumerate(labels):
    dur_real = real_stats[label]["mean_dur"]
    dur_sim  = np.mean(sim_mean_durs[r])
    dur_std  = np.std(sim_mean_durs[r])
    print(f"  Duración media {label:<12} {dur_real:12.2f} {dur_sim:12.2f} {dur_std:10.2f} {'':>8}")

print()
for r, label in enumerate(labels):
    frac_real = real_stats[label]["frac"]
    frac_sim  = np.mean(sim_fracs[r])
    frac_std  = np.std(sim_fracs[r])
    print(f"  Fracción en {label:<14} {frac_real:12.4f} {frac_sim:12.4f} {frac_std:10.4f} {pi_sp[r]:8.4f}")

max_bear_real = max(regime_durations(regimes_sp, 1))
print(f"\n  Max meses Bear consecutivos {max_bear_real:12d} "
      f"{np.mean(sim_max_bear):12.1f} {np.std(sim_max_bear):10.1f}")

# ── Histograma: máxima racha Bear ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(sim_max_bear, bins=30, color=COLORS["red"], alpha=0.6,
        edgecolor="white", label="Simulado (1000 trays.)")
ax.axvline(max_bear_real, color=COLORS["dark"], lw=2.5, ls="--",
           label=f"Observado = {max_bear_real}")
ax.set_title("Distribución de la máxima racha Bear consecutiva", fontsize=13)
ax.set_xlabel("Meses consecutivos en Bear")
ax.set_ylabel("Frecuencia")
ax.legend()
fig.tight_layout()
plt.show()


## Parte 4: Teorema ergódico aplicado a finanzas

El **teorema ergódico** afirma que, para una cadena irreducible y aperiódica,
las fracciones de tiempo convergen a la distribución estacionaria $\pi$.

Esto nos permite hacer predicciones de largo plazo:

$$\text{Rendimiento esperado a largo plazo} = \sum_{j} \pi_j \cdot \bar{r}_j$$

donde $\bar{r}_j$ es el rendimiento medio observado en el régimen $j$.


In [ ]:
# ── Rendimiento esperado de largo plazo ──────────────────────────────────
mean_returns_by_regime = np.array([
    returns_sp[regimes_sp == r].mean() for r in range(3)
])

expected_lr_return = pi_sp @ mean_returns_by_regime
naive_mean = returns_sp.mean()

print("TEOREMA ERGÓDICO — Predicciones de largo plazo (S&P 500)")
print("=" * 60)
print(f"\nRendimiento mensual medio por régimen:")
for r, label in enumerate(labels):
    print(f"  r̄({label:>4}) = {mean_returns_by_regime[r]:+.5f}")

print(f"\nRendimiento esperado (Σ πⱼ·r̄ⱼ) = {expected_lr_return:+.5f}")
print(f"Media histórica naive             = {naive_mean:+.5f}")
print(f"Diferencia                        = {abs(expected_lr_return - naive_mean):.6f}")

# ── Duraciones esperadas de régimen ──────────────────────────────────────
print(f"\n{'─' * 60}")
print(f"DURACIONES ESPERADAS DE RÉGIMEN")
print(f"{'─' * 60}")
print(f"\n{'Régimen':<10} {'P[j,j]':>8} {'E[duración]':>14} {'Obs. media':>14}")
print("-" * 50)

for r, label in enumerate(labels):
    p_self = P_sp[r, r]
    expected_dur = 1.0 / (1.0 - p_self)
    obs_durs = regime_durations(regimes_sp, r)
    obs_mean = np.mean(obs_durs) if obs_durs else 0
    print(f"{label:<10} {p_self:8.4f} {expected_dur:14.2f} {obs_mean:14.2f}")

print(f"\nNota: E[duración] = 1/(1 - P[j,j]) es la duración media")
print(f"de una racha en el régimen j bajo el modelo geométrico.")


> **Nota sobre sesgo de clasificación.** Los retornos promedio por régimen están calculados usando solo los meses clasificados en cada categoría por los umbrales fijos. Esto introduce un sesgo: el promedio "Alcista" solo incluye retornos > 2% (distribución truncada), lo cual infla la media respecto al verdadero retorno promedio durante un régimen alcista completo. Análogamente, el promedio "Bajista" solo incluye retornos < -2%. Los valores teóricos del teorema ergódico ($\sum \pi_j r_j$) heredan este sesgo. En la práctica, modelos más sofisticados (HMMs, filtros de Kalman) estiman los parámetros de cada régimen sin depender de umbrales fijos.

## Parte 5: Limitaciones del modelo de Markov para finanzas

El modelo de regímenes de Markov es una simplificación poderosa, pero tiene
limitaciones importantes:

1. **No estacionariedad** — Las condiciones del mercado cambian: la matriz de
   transición de los años 1960 no es la misma que la de los 2020.

2. **Propiedad sin memoria** — La propiedad de Markov asume que el futuro
   depende solo del estado actual, pero en finanzas existe **momentum**: un
   mercado que ha subido 6 meses seguidos tiene inercia distinta a uno que
   acaba de cambiar de régimen.

3. **Regímenes discretos** — Forzar tres categorías pierde información.
   Los rendimientos son continuos y la frontera entre Bull y Flat es arbitraria.

4. **Volatilidad variable** — La volatilidad cambia con el tiempo (clusters
   de volatilidad), algo que un modelo de Markov homogéneo no captura.


In [ ]:
# ── Prueba empírica de no estacionariedad ────────────────────────────────
half = n_sp // 2
regimes_first = regimes_sp[:half]
regimes_second = regimes_sp[half:]

P_first, _  = estimate_transition_matrix(regimes_first)
P_second, _ = estimate_transition_matrix(regimes_second)

print("EVIDENCIA DE NO ESTACIONARIEDAD — S&P 500")
print("=" * 60)
print(f"\nPrimera mitad  (meses 1–{half}, ≈ 1965–1995)")
print("-" * 42)
header = "         " + "".join(f"{l:>10}" for l in labels)
print(header)
for i, row_label in enumerate(labels):
    row = f"{row_label:>8} " + "".join(f"{P_first[i, j]:10.4f}" for j in range(3))
    print(row)

print(f"\nSegunda mitad (meses {half + 1}–{n_sp}, ≈ 1995–2025)")
print("-" * 42)
print(header)
for i, row_label in enumerate(labels):
    row = f"{row_label:>8} " + "".join(f"{P_second[i, j]:10.4f}" for j in range(3))
    print(row)

print(f"\nDiferencia (segunda - primera)")
print("-" * 42)
print(header)
diff = P_second - P_first
for i, row_label in enumerate(labels):
    row = f"{row_label:>8} " + "".join(f"{diff[i, j]:+10.4f}" for j in range(3))
    print(row)

print(f"\n‖ΔP‖_F (norma Frobenius) = {np.linalg.norm(diff, 'fro'):.4f}")
print("\nUna diferencia notable confirma que la matriz de transición")
print("NO es constante en el tiempo — el supuesto de estacionariedad se viola.")


## Hacia modelos más sofisticados

El modelo de regímenes de Markov es un **primer paso**. Para capturar la
complejidad real de los mercados, se extiende en varias direcciones:

| Extensión | Idea clave | Relación con Markov |
|:----------|:-----------|:-------------------|
| **HMM** (Hidden Markov Models) | Los regímenes son *ocultos*; solo observamos rendimientos | El régimen es un estado latente de Markov |
| **GARCH** | La volatilidad varía en el tiempo con memoria | Captura clusters de volatilidad que Markov ignora |
| **Regime-switching GARCH** | Combina HMM + GARCH | Cada régimen tiene su propio proceso de volatilidad |
| **Aprendizaje por refuerzo** | Un agente aprende a actuar dado el régimen | El mercado es el *environment* con transiciones de Markov |


## Resumen

| Concepto | Resultado clave |
|:---------|:---------------|
| Regímenes de mercado | Bull / Bear / Flat clasificados por umbrales de rendimiento |
| Matriz estimada $\hat{P}$ | Frecuencias relativas de transiciones observadas |
| Distribución estacionaria $\pi$ | Fracción de tiempo de largo plazo en cada régimen |
| Rendimiento ergódico | $\sum \pi_j \bar{r}_j$ ≈ media histórica |
| Duración esperada | $1/(1 - P_{jj})$ meses por racha |
| No estacionariedad | La matriz cambia entre décadas — limitación central |

**Conclusión**: Las cadenas de Markov ofrecen un marco analítico elegante para
entender la dinámica de regímenes en mercados financieros. El teorema ergódico
permite predicciones de largo plazo, pero la no estacionariedad del mundo real
exige modelos más ricos (HMM, GARCH) para aplicaciones prácticas.
